# Il primo modello



## Preparazione in Colab

1. Salva una copia del notebook nel tuo Drive.
2. Esegui la cella seguente e attendi «Preparazione completata».
3. Prosegui dall’alto verso il basso e completa gli esercizi.

La cella carica i dataset e i controlli dal repository del corso. Non devi installare Python né caricare manualmente i CSV. Se la sessione viene riavviata, esegui nuovamente la preparazione.

In [ ]:
from pathlib import Path
import hashlib
import importlib.util
from urllib.request import urlopen
from IPython.display import display, Markdown

course_source = 'https://raw.githubusercontent.com/prof-davidaffo/kaggle-learntools/master'
course_root = (Path("/content") if Path("/content").is_dir() else Path.home()) / "ai-lab-colab"
course_root.mkdir(parents=True, exist_ok=True)
bootstrap_path = course_root / "_bootstrap.py"
bootstrap_hash = '24c516651bf205b1c9511923ad4971fdca39a6248568d2b3aa46c198e64d1821'
if not bootstrap_path.is_file() or hashlib.sha256(bootstrap_path.read_bytes()).hexdigest() != bootstrap_hash:
    with urlopen(course_source + "/ai_lab/tools/colab_runtime.py", timeout=90) as response:
        bootstrap_data = response.read()
    if hashlib.sha256(bootstrap_data).hexdigest() != bootstrap_hash:
        raise RuntimeError("Course setup changed. Reopen the latest notebook from the course site.")
    bootstrap_path.write_bytes(bootstrap_data)
bootstrap_spec = importlib.util.spec_from_file_location("ai_lab_colab_bootstrap", bootstrap_path)
bootstrap = importlib.util.module_from_spec(bootstrap_spec)
bootstrap_spec.loader.exec_module(bootstrap)
bootstrap.download_runtime(course_source + "/ai_lab/colab/runtime.json", course_source, course_root)
bootstrap.setup(course_root, globals(), deep_learning=False)
display(Markdown("**Preparazione completata.** Puoi eseguire le celle della lezione."))


# Scegliere i dati per il modello
Il dataset contiene troppe variabili per analizzarle tutte insieme o stamparle comodamente. Come ridurle a un insieme comprensibile?

Per ora scegliamo poche variabili usando il ragionamento sul problema. In seguito potrai studiare tecniche statistiche per selezionarle automaticamente. Prima dobbiamo conoscere i nomi delle colonne: la proprietà **`columns`** del DataFrame li elenca nell'ultima riga del codice seguente.


In [ ]:
import pandas as pd

melbourne_file_path = '../input/melbourne-housing-snapshot/melb_data.csv'
melbourne_data = pd.read_csv(melbourne_file_path) 
melbourne_data.columns

In [ ]:
# Nei dati di Melbourne mancano alcuni valori: per certe case non sono state registrate tutte le variabili.
# Vedremo come gestire i valori mancanti in una lezione successiva.
# I dati Iowa non hanno valori mancanti nelle colonne utilizzate.
# Per ora scegliamo la soluzione più semplice: escludiamo le case con dati mancanti.
# Per ora non è necessario approfondire; il codice è:

# dropna elimina i valori mancanti; na significa "non disponibile".
melbourne_data = melbourne_data.dropna(axis=0)

Esistono molti modi per selezionare dati; il [corso pandas](https://www.kaggle.com/learn/pandas) li approfondisce. Qui useremo:

1. La notazione con il punto per selezionare il valore da prevedere.
2. Una lista di colonne per selezionare le caratteristiche di ingresso.

## Scegliere il target
La notazione con il punto estrae una colonna come **Series**, una struttura simile a un DataFrame con una sola colonna. Il valore da prevedere è il **prediction target**, o semplicemente *target*. Per convenzione lo chiamiamo **`y`**. Selezioniamo i prezzi delle case di Melbourne:


In [ ]:
y = melbourne_data.Price

# Scegliere le feature
Le colonne date in ingresso al modello sono le **feature**, cioè le caratteristiche utilizzate per prevedere il target. Qui sono i fattori che aiutano a stimare il prezzo di una casa. Talvolta si usano tutte le colonne tranne il target; in altri casi conviene usarne meno.

Partiamo con poche feature. Più avanti confronteremo modelli con insiemi diversi. Per selezionare più colonne passiamo una lista di nomi tra parentesi quadre. Ogni nome è una stringa, quindi richiede le virgolette:


In [ ]:
melbourne_features = ['Rooms', 'Bathroom', 'Landsize', 'Lattitude', 'Longtitude']

Per convenzione chiamiamo questi dati **`X`**.


In [ ]:
X = melbourne_data[melbourne_features]

Controlliamo i dati con `describe` e con `head`, che mostra le prime righe.


In [ ]:
X.describe()

In [ ]:
X.head()

L'ispezione dei dati è una parte importante del lavoro: spesso fa emergere anomalie da approfondire.


# Costruire il modello
Useremo **scikit-learn**, importata nel codice come **`sklearn`**, una libreria molto diffusa per modellare dati tabellari.

Il procedimento è:

* **Definire** il tipo di modello, per esempio un albero, e i suoi parametri.
* **Addestrare**, con `fit`, per apprendere gli schemi dai dati.
* **Prevedere**, con `predict`.
* **Valutare** la qualità delle previsioni.

Ecco come definire un albero e addestrarlo sulle feature e sul target:


In [ ]:
from sklearn.tree import DecisionTreeRegressor

# Definisci il modello; assegna un numero a random_state per ottenere risultati riproducibili.
melbourne_model = DecisionTreeRegressor(random_state=1)

# Addestra il modello.
melbourne_model.fit(X, y)

Molti algoritmi usano casualità durante l'addestramento. Impostare **`random_state`** rende riproducibile l'esecuzione: è una buona abitudine. Puoi scegliere un numero qualsiasi; non è una scelta che determina sistematicamente la qualità del modello.

Ora abbiamo un modello addestrato. In pratica vorremmo usarlo su case nuove, delle quali non conosciamo il prezzo. Per capire come funziona `predict`, iniziamo dalle prime righe dei dati di training.


In [ ]:
print("Making predictions for the following 5 houses:")
print(X.head())
print("The predictions are")
print(melbourne_model.predict(X.head()))

Prova nell'**[esercizio sul primo modello](https://prof-davidaffo.github.io/kaggle-learntools/corsi/machine_learning/ex3.html)**.
